In [ ]:
import pandas as pd
import numpy as np
import commons as c
import plotly.express as px
import plotly.graph_objects as go

# Get datasets

In [ ]:
csv_normal_path = 'results/dataframes/results_normal.csv'
df_normal = pd.read_csv(csv_normal_path, dtype=c.type_dict)

csv_equiv_path = 'results/dataframes/results_equiv.csv'
df_equiv = pd.read_csv(csv_equiv_path, dtype=c.type_dict)

In [ ]:
df = pd.concat([df_equiv, df_normal], ignore_index=True)
df['true_label'] = np.where(df['true_label'] == True, "non-equivalent", "equivalent")
df['predicted_label'] = np.where(df['predicted_label'] == True, "non-equivalent", "equivalent")

# Get Box Plots

In [ ]:
def add_threshold_line(fig, cat_range, threshold, threshold_value, color):
    fig.add_shape(
        type="line",
        x0=-0.5,
        x1=len(cat_range) - 0.5,
        y0=threshold_value,
        y1=threshold_value,
        line=dict(color=color, dash="dash"),
        xref="x",
        yref="y",
    )

In [ ]:
def print_box_plot(df, cat, hw, metric, output_folder, file_name):

    df = df.copy()  # Ensure it's a copy
    
    cat_range = sorted(df[cat].unique())  # Extract unique values from the column    
    cat_mapping = {category: idx for idx, category in enumerate(cat_range)}
    df['cat_numeric'] = df[cat].map(cat_mapping)  # Map categories to numeric values

    # Add an offset to the x-axis values based on 'true_label' to spread the points
    label_offsets = {
        "equivalent": -0.1,   # Adjust this value as needed
        "non-equivalent": 0.1  # Adjust this value as needed
    }
    
    # Apply the offset to a new column for the x-axis
    df['x_offset'] = df['cat_numeric'] + df['true_label'].map(label_offsets)

    label_mapping = {
        "equivalent": "Equivalent mutant",
        "non-equivalent": "Non-Equivalent mutant"
    }    
    df['true_label'] = df['true_label'].map(label_mapping)
    
    # Create the box plot with the adjusted x-axis values
    fig = px.box(
        df, 
        y="noisy_distance", 
        x="x_offset", 
        color="true_label", 
        category_orders={cat: cat_range},
        points=False
    ) 
    
    # Add the threshold line
    for t, color in zip(c.thresholds, px.colors.qualitative.Prism):
        threshold_value = c.get_tolerance_values(hw, t)[metric]
        add_threshold_line(fig, cat_range, t, threshold_value, color)
        
        # Add a dummy scatter trace for the legend
        fig.add_trace(go.Scatter(
            x=[None], y=[None],  # No actual points
            mode="lines",
            line=dict(color=color, dash="dash"),
            name=f"Threshold {t}",
            legendgroup="Thresholds",  # Group all thresholds together
            showlegend=True
        ))
    
    # Adjust layout for better visualization
    fig.update_layout(
        scattermode="group",
        #scattergap=0.75,
        xaxis=dict(
            title=cat, 
            categoryorder="array", 
            categoryarray=cat_range,
            tickvals=list(range(len(cat_range))),
            ticktext=cat_range
        ),
        yaxis_title="Distance between original and mutant",
        xaxis_title="Characteristic",
        legend_title_text= "Legend", #"Expected value",
        boxgroupgap=0, 
        boxgap=0
    )
    
    # Update legend to group "Thresholds" separately
    fig.update_layout(
        legend=dict(
            traceorder="normal",  # Keep order consistent
            title=dict(text="Legend"),  # Set legend title
        )
    )
    
    # Save the figure
    c.setup_layout_and_save(fig, "RQ2", output_folder, file_name, yaxis_range=[0, 1])


In [ ]:
def category_plot(df, m, metric):
    
    output_folder = f'results/RQ2/{m}'
    
    for hw in c.hardware:
        df_hw = df[df['hardware'] == hw]
        df_metric = df_hw[df_hw['metric'] == m]
        
        for key, categories in c.table_data.items():
            selected_columns = df_metric[[key, 'true_label', 'ideal_distance', 'noisy_distance']]
            file_name = f'{hw}_{key}'
            print_box_plot(selected_columns, key, hw, metric, output_folder, file_name)
            
        columns = ['Qubits_number', 'gates', 'depth', 'singlequbit_gates', 'multiqubit_gates'] 
        for cat in columns: 
            selected_columns = df_metric[[cat, 'true_label', 'ideal_distance', 'noisy_distance']]  
            file_name = f'{hw}_{cat}'
            print_box_plot(selected_columns, cat, hw, metric, output_folder, file_name)
    

In [ ]:
m = "T"
metric = "trace"
category_plot(df, m, metric)

m = "H"
metric = "hellinger"
category_plot(df, m, metric)